# Adversarially Robust DDoS Detection in SDN Networks
## Hybrid CNN-MLP with FGSM and PGD Adversarial Training

**Course:** Information Security  
**Semester:** 8th  

| Role | Name | Roll No. |
|------|------|----------|
| Author | *(your name)* | *(your roll no.)* |

---

### Project Summary

This notebook extends the baseline CNN-MLP model from **Mehmood et al. (2025)**,
which used SHAP feature selection and Bayesian hyperparameter optimization for DDoS
detection in SDN. The baseline was never evaluated under adversarial conditions.

We answer two research questions:
1. **How vulnerable** is the baseline CNN-MLP to FGSM and PGD adversarial attacks?
2. **How much robustness** can adversarial training recover — and at what cost to clean accuracy?

All models, metrics, and plots are saved to `/kaggle/working/`.

## 1. Setup: Dependencies and Imports

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Uncomment the line below when running on a fresh Kaggle kernel.
# !pip install -r /kaggle/working/info_sec_project/requirements.txt -q

# If you cloned the repo to /kaggle/working/, add it to sys.path so Python
# can find the `src` package and `config` module.
import sys
import os

REPO_ROOT = "/kaggle/working/info_sec_project"  # adjust if needed
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

In [ ]:
import logging
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Project modules
from config import CFG
from src.preprocessing import preprocess_pipeline
from src.feature_selection import (
    compute_shap_values,
    get_top_features,
    select_features,
    select_k_best_variance,
)
from src.model import build_model, train_clean, make_loader, predict
from src.attacks import fgsm_attack, pgd_attack, evaluate_under_attack
from src.adv_training import adversarial_train, combined_train
from src.evaluation import (
    compute_metrics,
    evaluate_model,
    robustness_curve,
    plot_robustness_curve,
    plot_confusion_matrix,
    comparison_table,
)

# Reproducibility
random.seed(CFG.RANDOM_SEED)
np.random.seed(CFG.RANDOM_SEED)
torch.manual_seed(CFG.RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.RANDOM_SEED)

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {CFG.DEVICE}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

## 2. Configuration

All hyperparameters live in `config.py`. Edit that file to change anything —
do **not** hardcode values below.

In [ ]:
# Print the full configuration for traceability in the notebook output
from dataclasses import asdict
cfg_dict = asdict(CFG)
# Convert Path objects to strings for display
cfg_display = {k: str(v) if isinstance(v, Path) else v for k, v in cfg_dict.items()}
print(json.dumps(cfg_display, indent=2))

## 3. Data Loading and Preprocessing

We use the CIC-IDS-2017 dataset (or any SDN traffic dataset) stored as a
Parquet file. The preprocessing pipeline:
1. Loads the raw file.
2. Drops non-feature columns (IPs, ports, Flow ID, Timestamp).
3. Replaces ±∞ with NaN and drops affected rows.
4. Binarises labels (0 = BENIGN, 1 = ATTACK).
5. Performs stratified 70/15/15 train/val/test split.
6. Applies Min-Max scaling (fit on train only to prevent leakage).

In [ ]:
# ─── Update DATA_PATH in config.py to point to your attached dataset ─────────
DATA_PATH = CFG.DATA_PATH
print(f"Loading data from: {DATA_PATH}")

data = preprocess_pipeline(DATA_PATH)

X_train, X_val, X_test = data["X_train"], data["X_val"], data["X_test"]
y_train, y_val, y_test = data["y_train"], data["y_val"], data["y_test"]
feature_names = data["feature_names"]
scaler = data["scaler"]

print(f"\nDataset shapes:")
print(f"  Train : {X_train.shape}, labels: {y_train.shape}")
print(f"  Val   : {X_val.shape},   labels: {y_val.shape}")
print(f"  Test  : {X_test.shape},  labels: {y_test.shape}")
print(f"  Features: {len(feature_names)}")

# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
print(f"\nTrain class distribution:")
for cls, cnt in zip(unique, counts):
    label = 'BENIGN' if cls == 0 else 'ATTACK'
    print(f"  {label} ({cls}): {cnt} ({cnt/len(y_train)*100:.1f}%)")

## 4. Baseline CNN-MLP Training (Clean Data)

Train the baseline model on all features using standard cross-entropy loss.
No adversarial examples involved at this stage.

In [ ]:
# Build DataLoaders
train_loader = make_loader(X_train, y_train, CFG.BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   CFG.BATCH_SIZE, shuffle=False)
test_loader  = make_loader(X_test,  y_test,  CFG.BATCH_SIZE, shuffle=False)

# Build model
num_features = X_train.shape[1]
baseline_model = build_model(num_features, num_classes=CFG.NUM_CLASSES, config=CFG)
baseline_model = baseline_model.to(CFG.DEVICE)
print(baseline_model)

# Train
baseline_model, baseline_history = train_clean(
    baseline_model, train_loader, val_loader,
    epochs=CFG.EPOCHS,
    lr=CFG.LEARNING_RATE,
    device=CFG.DEVICE,
    seed=CFG.RANDOM_SEED,
)
print("\nBaseline training complete.")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, CFG.EPOCHS + 1)

axes[0].plot(epochs_range, baseline_history["train_loss"], label="Train Loss", marker="o")
axes[0].plot(epochs_range, baseline_history["val_loss"],   label="Val Loss",   marker="s")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("Baseline CNN-MLP: Loss Curves")
axes[0].legend()

axes[1].plot(epochs_range, baseline_history["train_acc"], label="Train Acc", marker="o")
axes[1].plot(epochs_range, baseline_history["val_acc"],   label="Val Acc",   marker="s")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Baseline CNN-MLP: Accuracy Curves")
axes[1].legend()
axes[1].set_ylim(0, 1.05)

fig.tight_layout()
curve_path = CFG.RESULTS_DIR / "baseline_training_curves.png"
fig.savefig(curve_path, dpi=CFG.PLOT_DPI, bbox_inches="tight")
plt.show()
print(f"Saved → {curve_path}")

## 5. Baseline Model Evaluation on Clean Test Data

In [ ]:
baseline_clean_metrics, y_true, y_pred_baseline = evaluate_model(
    baseline_model, test_loader, CFG.DEVICE
)

print("Baseline CNN-MLP — Clean Test Metrics:")
for k, v in baseline_clean_metrics.items():
    print(f"  {k:12s}: {v:.4f}")

# Confusion matrix
cm_path = CFG.RESULTS_DIR / "baseline_confusion_matrix.png"
plot_confusion_matrix(
    y_true, y_pred_baseline,
    save_path=cm_path,
    title="Baseline CNN-MLP — Clean Test Set",
    dpi=CFG.PLOT_DPI,
)
print(f"Confusion matrix saved → {cm_path}")

## 6. SHAP Feature Selection

SHAP (SHapley Additive exPlanations) values are computed using the trained
baseline model. The top-20 features by mean |SHAP value| are retained.

> **Note:** SHAP requires a trained model and runs post-training. If DeepExplainer
> fails due to unsupported operations, KernelExplainer is used as fallback.
> The variance-based fallback (`select_k_best_variance`) is available if SHAP
> is too slow.

In [ ]:
# Use a small background set from training data
SHAP_BG_SIZE = 300
bg_idx = np.random.choice(len(X_train), SHAP_BG_SIZE, replace=False)
X_background = X_train[bg_idx]

# Explain on a validation subset
shap_values = compute_shap_values(
    model=baseline_model,
    X_background=X_background,
    X_explain=X_val,
    device=CFG.DEVICE,
    max_samples=1000,
)

top_features = get_top_features(shap_values, feature_names, k=CFG.NUM_FEATURES)
print(f"\nTop-{CFG.NUM_FEATURES} features selected by SHAP:")
for i, f in enumerate(top_features, 1):
    print(f"  {i:2d}. {f}")

In [ ]:
# SHAP bar plot — mean absolute SHAP values
import shap
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx = [feature_names.index(f) for f in top_features]
top_shap = mean_abs_shap[top_idx]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_features[::-1], top_shap[::-1], color="steelblue")
ax.set_xlabel("Mean |SHAP Value|", fontsize=12)
ax.set_title(f"Top-{CFG.NUM_FEATURES} Feature Importances (SHAP)", fontsize=13)
fig.tight_layout()
shap_fig_path = CFG.RESULTS_DIR / "shap_feature_importance.png"
fig.savefig(shap_fig_path, dpi=CFG.PLOT_DPI, bbox_inches="tight")
plt.show()
print(f"SHAP plot saved → {shap_fig_path}")

# Reduce feature sets to top-k
X_train_sel = select_features(X_train, feature_names, top_features)
X_val_sel   = select_features(X_val,   feature_names, top_features)
X_test_sel  = select_features(X_test,  feature_names, top_features)
print(f"\nReduced feature shape: {X_train_sel.shape}")

In [ ]:
# Optional: retrain baseline on the top-k features
# This matches the Mehmood et al. pipeline more closely.

train_loader_sel = make_loader(X_train_sel, y_train, CFG.BATCH_SIZE, shuffle=True)
val_loader_sel   = make_loader(X_val_sel,   y_val,   CFG.BATCH_SIZE, shuffle=False)
test_loader_sel  = make_loader(X_test_sel,  y_test,  CFG.BATCH_SIZE, shuffle=False)

num_features_sel = X_train_sel.shape[1]
baseline_model_sel = build_model(num_features_sel, num_classes=CFG.NUM_CLASSES, config=CFG)
baseline_model_sel = baseline_model_sel.to(CFG.DEVICE)

baseline_model_sel, baseline_history_sel = train_clean(
    baseline_model_sel, train_loader_sel, val_loader_sel,
    epochs=CFG.EPOCHS,
    lr=CFG.LEARNING_RATE,
    device=CFG.DEVICE,
    seed=CFG.RANDOM_SEED,
)

baseline_sel_metrics, _, y_pred_sel = evaluate_model(
    baseline_model_sel, test_loader_sel, CFG.DEVICE
)
print("\nBaseline (top-k SHAP features) — Clean Test Metrics:")
for k, v in baseline_sel_metrics.items():
    print(f"  {k:12s}: {v:.4f}")

## 7. Adversarial Attack Generation and Baseline Vulnerability Assessment

We evaluate the **baseline model** (trained on selected features) under FGSM
and PGD attacks to establish how much clean accuracy degrades.

Both attacks use the L∞ threat model and are clamped to [0, 1] since features
are Min-Max scaled.

In [ ]:
FGSM_KWARGS = {"epsilon": CFG.FGSM_EPSILON}
PGD_KWARGS  = {
    "epsilon": CFG.PGD_EPSILON,
    "alpha": CFG.PGD_ALPHA,
    "num_steps": CFG.PGD_STEPS,
}

print("Evaluating baseline under FGSM ...")
baseline_fgsm_acc = evaluate_under_attack(
    baseline_model_sel, test_loader_sel,
    fgsm_attack, FGSM_KWARGS, CFG.DEVICE
)

print("\nEvaluating baseline under PGD ...")
baseline_pgd_acc = evaluate_under_attack(
    baseline_model_sel, test_loader_sel,
    pgd_attack, PGD_KWARGS, CFG.DEVICE
)

baseline_fgsm_metrics = {"accuracy": baseline_fgsm_acc}
baseline_pgd_metrics  = {"accuracy": baseline_pgd_acc}

print(f"\nBaseline accuracy summary:")
print(f"  Clean : {baseline_sel_metrics['accuracy']:.4f}")
print(f"  FGSM  : {baseline_fgsm_acc:.4f}  (ε={CFG.FGSM_EPSILON})")
print(f"  PGD   : {baseline_pgd_acc:.4f}   (ε={CFG.PGD_EPSILON}, steps={CFG.PGD_STEPS})")

## 8. Adversarial Training (PGD-based Robust Model)

We train a **robust model** using PGD adversarial training. Each mini-batch:
1. Computes clean loss on the original batch.
2. Generates PGD adversarial examples.
3. Computes adversarial loss.
4. Combines: `L_total = L_clean + w * L_adv`

This is the primary defence — the model learns decision boundaries that are
resistant to gradient-based perturbations.

In [ ]:
robust_model = build_model(num_features_sel, num_classes=CFG.NUM_CLASSES, config=CFG)
robust_model = robust_model.to(CFG.DEVICE)

robust_model, robust_history = adversarial_train(
    model=robust_model,
    train_loader=train_loader_sel,
    val_loader=val_loader_sel,
    attack_type="pgd",
    epochs=CFG.EPOCHS,
    lr=CFG.LEARNING_RATE,
    attack_kwargs=PGD_KWARGS,
    adv_loss_weight=CFG.ADV_LOSS_WEIGHT,
    device=CFG.DEVICE,
    seed=CFG.RANDOM_SEED,
)
print("\nPGD adversarial training complete.")

In [ ]:
# Plot adversarial training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, CFG.EPOCHS + 1)

axes[0].plot(ep, robust_history["train_loss"], label="Train Loss", marker="o")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("PGD Adversarial Training: Loss")
axes[0].legend()

axes[1].plot(ep, robust_history["train_clean_acc"], label="Train Clean",   marker="o")
axes[1].plot(ep, robust_history["train_adv_acc"],   label="Train Adv",     marker="s")
axes[1].plot(ep, robust_history["val_clean_acc"],   label="Val Clean",     marker="^", linestyle="--")
axes[1].plot(ep, robust_history["val_adv_acc"],     label="Val Adv (PGD)", marker="D", linestyle="--")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("PGD Adversarial Training: Accuracy")
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1.05)

fig.tight_layout()
adv_curve_path = CFG.RESULTS_DIR / "adv_training_curves.png"
fig.savefig(adv_curve_path, dpi=CFG.PLOT_DPI, bbox_inches="tight")
plt.show()
print(f"Saved → {adv_curve_path}")

## 9. Robust Model Evaluation (Clean + Adversarial)

In [ ]:
robust_clean_metrics, y_true, y_pred_robust = evaluate_model(
    robust_model, test_loader_sel, CFG.DEVICE
)

print("Robust model — Clean Test Metrics:")
for k, v in robust_clean_metrics.items():
    print(f"  {k:12s}: {v:.4f}")

print("\nEvaluating robust model under FGSM ...")
robust_fgsm_acc = evaluate_under_attack(
    robust_model, test_loader_sel, fgsm_attack, FGSM_KWARGS, CFG.DEVICE
)

print("\nEvaluating robust model under PGD ...")
robust_pgd_acc = evaluate_under_attack(
    robust_model, test_loader_sel, pgd_attack, PGD_KWARGS, CFG.DEVICE
)

print(f"\nRobust model accuracy summary:")
print(f"  Clean : {robust_clean_metrics['accuracy']:.4f}")
print(f"  FGSM  : {robust_fgsm_acc:.4f}")
print(f"  PGD   : {robust_pgd_acc:.4f}")

# Confusion matrix for robust model
cm_robust_path = CFG.RESULTS_DIR / "robust_confusion_matrix.png"
plot_confusion_matrix(
    y_true, y_pred_robust,
    save_path=cm_robust_path,
    title="Robust CNN-MLP (PGD-trained) — Clean Test Set",
    dpi=CFG.PLOT_DPI,
)
print(f"Saved → {cm_robust_path}")

## 10. Robustness Curve: Accuracy vs ε

We sweep ε from 0.01 to 0.1 and measure accuracy under PGD for both the
baseline and robust models. A steeper drop in the baseline line confirms
the vulnerability, while a flatter robust line quantifies the gain.

In [ ]:
print("Computing robustness curve for BASELINE under PGD ...")
baseline_pgd_curve = robustness_curve(
    model=baseline_model_sel,
    data_loader=test_loader_sel,
    attack_fn=pgd_attack,
    epsilons=CFG.EPSILONS_TO_TEST,
    device=CFG.DEVICE,
    alpha=CFG.PGD_ALPHA,
    num_steps=CFG.PGD_STEPS,
)

print("\nComputing robustness curve for ROBUST model under PGD ...")
robust_pgd_curve = robustness_curve(
    model=robust_model,
    data_loader=test_loader_sel,
    attack_fn=pgd_attack,
    epsilons=CFG.EPSILONS_TO_TEST,
    device=CFG.DEVICE,
    alpha=CFG.PGD_ALPHA,
    num_steps=CFG.PGD_STEPS,
)

rc_path = CFG.RESULTS_DIR / "robustness_curve_pgd.png"
plot_robustness_curve(
    results_dict={
        "Baseline CNN-MLP": baseline_pgd_curve,
        "PGD-Robust CNN-MLP": robust_pgd_curve,
    },
    save_path=rc_path,
    dpi=CFG.PLOT_DPI,
)
print(f"Robustness curve saved → {rc_path}")

## 11. Ablation Study

We compare three adversarial training variants:

| Variant | Description |
|---------|-------------|
| A | FGSM-only adversarial training |
| B | PGD-only adversarial training (already done above) |
| C | Combined FGSM + PGD adversarial training |

Each variant is evaluated on clean, FGSM, and PGD test sets.

In [ ]:
# ── Variant A: FGSM-only adversarial training ─────────────────────────────────
print("Training Variant A: FGSM-only ...")
model_a = build_model(num_features_sel, num_classes=CFG.NUM_CLASSES, config=CFG).to(CFG.DEVICE)

model_a, history_a = adversarial_train(
    model=model_a,
    train_loader=train_loader_sel,
    val_loader=val_loader_sel,
    attack_type="fgsm",
    epochs=CFG.EPOCHS,
    lr=CFG.LEARNING_RATE,
    attack_kwargs=FGSM_KWARGS,
    adv_loss_weight=CFG.ADV_LOSS_WEIGHT,
    device=CFG.DEVICE,
    seed=CFG.RANDOM_SEED,
)
print("Variant A done.")

In [ ]:
# ── Variant C: Combined FGSM + PGD adversarial training ──────────────────────
print("Training Variant C: Combined FGSM + PGD ...")
model_c = build_model(num_features_sel, num_classes=CFG.NUM_CLASSES, config=CFG).to(CFG.DEVICE)

model_c, history_c = combined_train(
    model=model_c,
    train_loader=train_loader_sel,
    val_loader=val_loader_sel,
    epochs=CFG.EPOCHS,
    lr=CFG.LEARNING_RATE,
    fgsm_kwargs=FGSM_KWARGS,
    pgd_kwargs=PGD_KWARGS,
    adv_loss_weight=CFG.ADV_LOSS_WEIGHT,
    device=CFG.DEVICE,
    seed=CFG.RANDOM_SEED,
)
print("Variant C done.")

In [ ]:
# ── Evaluate all variants ─────────────────────────────────────────────────────
ablation_results = {}

for name, mdl in [
    ("Baseline",        baseline_model_sel),
    ("FGSM-only (A)",   model_a),
    ("PGD-only (B)",    robust_model),
    ("Combined (C)",    model_c),
]:
    clean_m, _, _  = evaluate_model(mdl, test_loader_sel, CFG.DEVICE)
    fgsm_acc = evaluate_under_attack(mdl, test_loader_sel, fgsm_attack, FGSM_KWARGS, CFG.DEVICE)
    pgd_acc  = evaluate_under_attack(mdl, test_loader_sel, pgd_attack,  PGD_KWARGS,  CFG.DEVICE)

    ablation_results[name] = {
        "Clean Acc":  round(clean_m["accuracy"], 4),
        "Clean F1":   round(clean_m["f1"], 4),
        "FGSM Acc":   round(fgsm_acc, 4),
        "PGD Acc":    round(pgd_acc, 4),
    }

ablation_df = pd.DataFrame(ablation_results).T
print("\nAblation Study Results:")
print(ablation_df.to_string())

ablation_csv_path = CFG.RESULTS_DIR / "ablation_study.csv"
ablation_df.to_csv(ablation_csv_path)
print(f"\nSaved → {ablation_csv_path}")

In [ ]:
# Robustness curves for all four variants under PGD
all_curves = {}
for name, mdl in [
    ("Baseline",      baseline_model_sel),
    ("FGSM-only (A)", model_a),
    ("PGD-only (B)",  robust_model),
    ("Combined (C)",  model_c),
]:
    print(f"Computing robustness curve for {name} ...")
    all_curves[name] = robustness_curve(
        model=mdl,
        data_loader=test_loader_sel,
        attack_fn=pgd_attack,
        epsilons=CFG.EPSILONS_TO_TEST,
        device=CFG.DEVICE,
        alpha=CFG.PGD_ALPHA,
        num_steps=CFG.PGD_STEPS,
    )

ablation_rc_path = CFG.RESULTS_DIR / "ablation_robustness_curves.png"
plot_robustness_curve(
    results_dict=all_curves,
    save_path=ablation_rc_path,
    dpi=CFG.PLOT_DPI,
)
print(f"Ablation robustness curves saved → {ablation_rc_path}")

## 12. Final Comparison Table

Summary table comparing the **baseline** vs the **best robust model** across
clean, FGSM, and PGD conditions. This is the central result for the report.

In [ ]:
# Get full metrics for each condition (baseline)
baseline_results = {
    "clean": baseline_sel_metrics,
    "fgsm":  {"accuracy": baseline_fgsm_acc, "f1": float("nan")},
    "pgd":   {"accuracy": baseline_pgd_acc,  "f1": float("nan")},
}

robust_results = {
    "clean": robust_clean_metrics,
    "fgsm":  {"accuracy": robust_fgsm_acc, "f1": float("nan")},
    "pgd":   {"accuracy": robust_pgd_acc,  "f1": float("nan")},
}

csv_path = CFG.RESULTS_DIR / "final_comparison.csv"
df_comparison = comparison_table(baseline_results, robust_results, csv_path)
print("\nFinal Comparison Table:")
print(df_comparison.to_string())

## 13. Save All Models and Results

All model weights are saved as `.pth` files and all metric dictionaries
are serialised as JSON to `/kaggle/working/models/` and
`/kaggle/working/results/` respectively.

In [ ]:
# ── Save model weights ────────────────────────────────────────────────────────
models_to_save = {
    "baseline_all_features":    baseline_model,
    "baseline_shap_features":   baseline_model_sel,
    "robust_pgd":               robust_model,
    "robust_fgsm_only":         model_a,
    "robust_combined":          model_c,
}

for name, mdl in models_to_save.items():
    save_path = CFG.MODELS_DIR / f"{name}.pth"
    torch.save(mdl.state_dict(), save_path)
    print(f"Saved model → {save_path}")

# ── Save training histories ───────────────────────────────────────────────────
histories = {
    "baseline_history_all":     baseline_history,
    "baseline_history_sel":     baseline_history_sel,
    "robust_pgd_history":       robust_history,
    "robust_fgsm_history":      history_a,
    "robust_combined_history":  history_c,
}

for name, hist in histories.items():
    save_path = CFG.RESULTS_DIR / f"{name}.json"
    with open(save_path, "w") as f:
        json.dump(hist, f, indent=2)
    print(f"Saved history → {save_path}")

# ── Save top features list ────────────────────────────────────────────────────
top_features_path = CFG.RESULTS_DIR / "top_shap_features.json"
with open(top_features_path, "w") as f:
    json.dump(top_features, f, indent=2)
print(f"Saved top features → {top_features_path}")

# ── Save robustness curves ────────────────────────────────────────────────────
rc_json_path = CFG.RESULTS_DIR / "robustness_curves.json"
# Convert float keys to strings for JSON serialisation
serialisable_curves = {
    name: {str(eps): acc for eps, acc in curve.items()}
    for name, curve in all_curves.items()
}
with open(rc_json_path, "w") as f:
    json.dump(serialisable_curves, f, indent=2)
print(f"Saved robustness curves → {rc_json_path}")

print("\nAll files saved successfully.")
print(f"  Models  : {CFG.MODELS_DIR}")
print(f"  Results : {CFG.RESULTS_DIR}")